# Serialized Table QA with a Relational GNN Residual

This notebook compares the serialized LoRA baseline, the best CNN residual result, and a relational GNN residual. The GNN uses typed row, column, and header edges. All run artifacts are written directly to Google Drive and resume after a disconnect.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys

REPO_URL = "https://github.com/seungjun-green/cnn_qwen_table_mcr.git"
REPO_DIR = Path("/content/table-cnn-mrc")
if (REPO_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
elif REPO_DIR.exists():
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository")
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")],
    check=True,
)
%cd /content/table-cnn-mrc

In [ ]:
from google.colab import drive, userdata

drive.mount("/content/drive")
try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token

DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/cnn_qwen_table_mcr/outputs")
DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"All run files will be saved directly under: {DRIVE_OUTPUT_ROOT}")

## Select experiments

All three comparison runs are selected by default. Existing completed or early-stopped runs exit immediately, so normally only the new GNN model will train.

In [ ]:
CONFIGURATIONS_TO_RUN = "all"

AVAILABLE_CONFIGURATIONS = [
    "serialized_table_lora",
    "cnn_residual_mean_early",
    "gnn_residual_relational_early",
]
requested = [name.strip() for name in CONFIGURATIONS_TO_RUN.splitlines() if name.strip()]
selected_configurations = AVAILABLE_CONFIGURATIONS if requested == ["all"] else requested
unknown = sorted(set(selected_configurations) - set(AVAILABLE_CONFIGURATIONS))
if unknown:
    raise ValueError(f"Unknown configuration names: {unknown}")
if not selected_configurations:
    raise ValueError("Select at least one configuration")
print("Selected configurations:")
for position, name in enumerate(selected_configurations, start=1):
    print(f"  {position}. {name}")

## GNN smoke test

Run this before training. It checks serialized-token alignment, typed graph construction, loss, gradients, and generation on one real example.

In [ ]:
GNN_CONFIG = REPO_DIR / "configs/gnn_residual_relational_early.yaml"
smoke_command = [
    sys.executable, "-u", str(REPO_DIR / "scripts/smoke_test.py"),
    "--config", str(GNN_CONFIG),
]
smoke_command_text = " ".join(shlex.quote(str(part)) for part in smoke_command)
smoke_status_path = Path("/tmp/table_mrc_gnn_smoke_exit_code.txt")
smoke_status_path.unlink(missing_ok=True)
get_ipython().system(
    f"cd {shlex.quote(str(REPO_DIR))} && PYTHONUNBUFFERED=1 "
    f"{smoke_command_text}; printf '%s' $? > {shlex.quote(str(smoke_status_path))}"
)
if not smoke_status_path.is_file():
    raise RuntimeError("GNN smoke-test exit status was not recorded")
smoke_return_code = int(smoke_status_path.read_text(encoding="utf-8").strip())
if smoke_return_code != 0:
    raise subprocess.CalledProcessError(smoke_return_code, smoke_command)

## Train or resume

The child process uses compact periodic logs instead of `tqdm`, avoiding repeated Colab output rows. Checkpoints are saved directly to Drive every 100 optimizer steps.

In [ ]:
for position, name in enumerate(selected_configurations, start=1):
    config_path = REPO_DIR / "configs" / f"{name}.yaml"
    drive_output = DRIVE_OUTPUT_ROOT / name
    drive_output.mkdir(parents=True, exist_ok=True)
    separator = "#" * 88
    print(f"\n{separator}", flush=True)
    print(f"CONFIGURATION {position}/{len(selected_configurations)}: {name}", flush=True)
    print(f"DIRECT DRIVE OUTPUT: {drive_output}", flush=True)
    print(f"{separator}\n", flush=True)
    command = [
        sys.executable,
        "-u",
        str(REPO_DIR / "scripts/run_experiment.py"),
        "--config",
        str(config_path),
        "--output-dir",
        str(drive_output),
    ]
    command_text = " ".join(shlex.quote(str(part)) for part in command)
    status_path = Path("/tmp") / f"table_mrc_{name}_exit_code.txt"
    status_path.unlink(missing_ok=True)
    shell_command = (
        f"cd {shlex.quote(str(REPO_DIR))} && "
        f"PYTHONUNBUFFERED=1 TABLE_MRC_PLAIN_PROGRESS=1 "
        f"TQDM_DISABLE=1 {command_text}; "
        f"printf '%s' $? > {shlex.quote(str(status_path))}"
    )
    get_ipython().system(shell_command)
    if not status_path.is_file():
        raise RuntimeError(f"Training exit status was not recorded for {name}")
    return_code = int(status_path.read_text(encoding="utf-8").strip())
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

## Compare results

In [ ]:
import json
import pandas as pd
from IPython.display import display

rows = []
for name in AVAILABLE_CONFIGURATIONS:
    history_path = DRIVE_OUTPUT_ROOT / name / "history.json"
    if not history_path.is_file():
        continue
    with history_path.open(encoding="utf-8") as handle:
        history = json.load(handle)
    epochs = history.get("epochs", [])
    rows.append({
        "configuration": name,
        "status": history.get("status"),
        "epochs completed": len(epochs),
        "primary metric": history.get("primary_metric"),
        "best validation score": history.get("best_metric"),
        "latest training loss": epochs[-1].get("training_loss") if epochs else None,
    })
results = pd.DataFrame(rows)
if not results.empty:
    results = results.sort_values("best validation score", ascending=False)
display(results)